# Guided Assignment: Product Details Tool

Make a tool, give it to a model, and see the model **ask** for it instead of answering.

In Colab, add your OpenRouter key under **Secrets** (key icon in the left sidebar) with the name `OPENROUTER_API_KEY`. Elsewhere, keep a `.env` file with `OPENROUTER_API_KEY` in this folder.

In [1]:
%pip install -q langchain_openai langchain_core python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Colab: read the key from Secrets. Elsewhere: read it from the .env file
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    import dotenv
    dotenv.load_dotenv(".env")

# The model, through OpenRouter
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    model_name="openai/gpt-4o-mini-2024-07-18",
)

# A small product catalogue (illustrative)
PRODUCTS = {
    "P101": {"name": "Wireless mouse", "price": 799, "description": "2.4 GHz, silent clicks, 12-month battery"},
    "P102": {"name": "Steel water bottle", "price": 549, "description": "1 litre, keeps drinks cold for 24 hours"},
    "P103": {"name": "Yoga mat", "price": 1199, "description": "6 mm thick, non-slip, comes with a strap"},
}

In [4]:
# Step 1: Write a tool get_product(product_id: str) -> dict with @tool on top.
# Add a docstring saying what it returns. Return PRODUCTS[product_id]

@tool
def get_product(product_id: str) -> dict:
    """Returns the product details for the given product ID."""
    if product_id not in PRODUCTS:
        raise ValueError(f"Product ID {product_id} not found.");

    return PRODUCTS[product_id]



In [5]:
# Step 2: Run the tool yourself with get_product.invoke() for product "P101"
print(get_product.invoke("P101"))

{'name': 'Wireless mouse', 'price': 799, 'description': '2.4 GHz, silent clicks, 12-month battery'}


In [6]:
# Step 3: Attach the tool to the model with bind_tools and call it llm_with_tools
llm_with_tools = llm.bind_tools([get_product])

In [7]:
# Step 4: Ask llm_with_tools "Tell me about product P103" and print the reply's tool_calls
llm_reply = llm_with_tools.invoke("Tell me about product P103")
print(llm_reply.tool_calls)

[{'name': 'get_product', 'args': {'product_id': 'P103'}, 'id': 'call_OVzUGXhq5lObNkKpZLF1hztc', 'type': 'tool_call'}]


In [10]:
print("Tool calls:", llm_reply.tool_calls)

Tool calls: [{'name': 'get_product', 'args': {'product_id': 'P103'}, 'id': 'call_OVzUGXhq5lObNkKpZLF1hztc', 'type': 'tool_call'}]


In [9]:
# Step 5: The model only asked. Run the call it asked for: pass its args to get_product.invoke()

first_call = llm_reply.tool_calls[0]
call_args = first_call["args"] if isinstance(first_call, dict) else first_call.args
product_id = call_args["product_id"]
product_details = get_product.invoke(product_id)
product_details

{'name': 'Yoga mat',
 'price': 1199,
 'description': '6 mm thick, non-slip, comes with a strap'}